# 🎬 OpenSource Clipping — Optimized Colab Notebook

**3-Cell Architecture:** ① Setup → ② Config (with input prompts) → ③ Runtime

**Features:**
- Interactive `input()` prompts for all key settings
- **Ollama** support (local/remote/cloud via OpenAI-compatible API)
- **YouTube Transcript API** (skip Whisper when available)
- **Per-clip overrides** (different ratio/font/hook/broll per rank)
- **VRAM monitoring** + aggressive cache clearing between stages
- **Drive model caching** (persist across sessions)

## ① SETUP — Mount Drive, Clone Repo, Install, Verify

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ═══════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil
from pathlib import Path

# ── 1.1 Mount Google Drive (for persistent model cache) ──
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ── 1.2 System diagnostics ──
def print_sys_info():
    print("═" * 60)
    try:
        gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.used,memory.total","--format=csv,noheader,nounits"],
                             capture_output=True, text=True, check=False)
        name, used, total = gpu.stdout.strip().split(", ")
        print(f"🎮 GPU : {name} | VRAM {float(used):.0f}/{float(total):.0f} MB")
    except Exception as e:
        print(f"⚠️ GPU read error: {e}")
    try:
        mem = dict((i.split()[0].rstrip(":"), int(i.split()[1])) for i in open("/proc/meminfo").read().splitlines())
        print(f"🧠 RAM : {mem.get('MemAvailable',mem.get('MemFree',0))/1024**2:.1f} GB free / {mem.get('MemTotal',0)/1024**2:.1f} GB total")
    except Exception as e:
        print(f"⚠️ RAM read error: {e}")
    stat = shutil.disk_usage("/content")
    print(f"💾 Disk: {stat.free/1024**3:.1f} GB free / {stat.total/1024**3:.1f} GB total")
    print("═" * 60)

print_sys_info()

# ── 1.3 Clone / update repo ──
REPO_URL = "https://github.com/troublescope/Ashortai.git"
CLONE_DIR = "/content/opensource-clipping"

if os.path.isdir(f"{CLONE_DIR}/.git"):
    print("📁 Repo exists — pulling latest…")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only"], check=True)
else:
    print("⬇️  Cloning repo…")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)

os.chdir(CLONE_DIR)

# ── 1.4 Install dependencies ──
print("⏳ Installing dependencies (this takes ~2-3 min)…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# ── Patch main.py: GOOGLE_API_KEY only required for Gemini provider ──
main_py = Path(CLONE_DIR) / "main.py"
main_text = main_py.read_text(encoding="utf-8")
old_check = '    if not cfg.api_key_gemini:\n        print("❌ ERROR: GOOGLE_API_KEY environment variable tidak ditemukan.")\n        print("   Set via: export GOOGLE_API_KEY=\'your-key\' atau buat file .env")\n        sys.exit(1)'
new_check = '    if cfg.ai_provider == "gemini" and not cfg.api_key_gemini:\n        print("❌ ERROR: GOOGLE_API_KEY environment variable tidak ditemukan.")\n        print("   Set via: export GOOGLE_API_KEY=\'your-key\' atau buat file .env")\n        sys.exit(1)'
if old_check in main_text and new_check not in main_text:
    main_py.write_text(main_text.replace(old_check, new_check), encoding="utf-8")
    print("🩹 Patched main.py: GOOGLE_API_KEY only required when provider is gemini.")
print("✅ Dependencies installed.")

# ── 1.5 Verify torch CUDA ──
import torch
print(f"🔥 PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")

# ── 1.6 Restore cached models from Drive ──
DRIVE_CACHE = Path("/content/drive/MyDrive/osc_cache")
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
MODELS = ["blaze_face_full_range.tflite", "face_yolov8m.pt", "face_yolov8n.pt", "face_yolov8s.pt"]
for m in MODELS:
    src, dst = DRIVE_CACHE / m, Path(CLONE_DIR) / m
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print(f"📥 Restored {m} from Drive cache")

print_sys_info()

## ② CONFIG — Interactive Input Prompts

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION (answer the prompts below)
# ═══════════════════════════════════════════════════════════════
import json
from pathlib import Path

def ask(prompt, default="", required=True):
    val = input(f"{prompt} [default: {default}]: ").strip() if default else input(f"{prompt}: ").strip()
    if not val and default:
        return str(default)
    if not val and required:
        raise ValueError(f"'{prompt}' is required.")
    return val

def ask_bool(prompt, default=True):
    hint = "Y/n" if default else "y/N"
    val = input(f"{prompt} [{hint}]: ").strip().lower()
    if not val:
        return default
    return val in ("y", "yes", "true", "1")

def ask_int(prompt, default, min_val=None, max_val=None):
    val = input(f"{prompt} [default: {default}]: ").strip()
    if not val:
        return int(default)
    v = int(val)
    if min_val is not None and v < min_val:
        raise ValueError(f"Value must be >= {min_val}")
    if max_val is not None and v > max_val:
        raise ValueError(f"Value must be <= {max_val}")
    return v

print("📝 Please answer the prompts below (press Enter to accept defaults):\n")

# ── API Keys ──
print("── API Keys ──")
GOOGLE_API_KEY = ask("GOOGLE_API_KEY (Gemini)", "", required=True)
HF_TOKEN       = ask("HF_TOKEN (optional, for split-screen)", "", required=False)
PEXELS_API_KEY = ask("PEXELS_API_KEY (optional, for B-roll)", "", required=False)
NVIDIA_API_KEY = ask("NVIDIA_API_KEY (optional)", "", required=False)
OLLAMA_API_KEY = ask("OLLAMA_API_KEY (optional, for cloud Ollama)", "", required=False)

env_text = f"""GOOGLE_API_KEY={GOOGLE_API_KEY}
HF_TOKEN={HF_TOKEN}
PEXELS_API_KEY={PEXELS_API_KEY}
NVIDIA_API_KEY={NVIDIA_API_KEY}
OLLAMA_API_KEY={OLLAMA_API_KEY}
"""
Path(".env").write_text(env_text, encoding="utf-8")
print("🔐 .env written successfully.\n")

# ── Source Video ──
print("── Source Video ──")
URL_VIDEO       = ask("Video URL", "https://www.youtube.com/watch?v=Dc4_aBFAYWE")
SOURCE_PLATFORM = ask("Platform (youtube/tiktok/instagram/gdrive)", "youtube")
SOURCE_HEIGHT   = ask("Source height (max/1080/1440/2160)", "max")

# ── Output ──
print("\n── Output ──")
JUMLAH_CLIP   = ask_int("Number of clips", 5, 1, 10)
RASIO         = ask("Aspect ratio (9:16/16:9/1:1/3:4/4:5)", "9:16")
RENDER_HEIGHT = ask("Render height (1080/1440/2160/source)", "1080")

# ── AI Provider ──
print("\n── AI Provider ──")
AI_PROVIDER = ask("Provider (gemini/nvidia/ollama)", "gemini")
GEMINI_MODEL  = ask("Gemini model", "gemini-3-flash-preview")
OLLAMA_URL    = ask("Ollama base URL", "https://ollama.com")
OLLAMA_MODEL  = ask("Ollama model", "gemini-3-flash-preview:cloud")
OLLAMA_FALLBACK_URL   = ask("Ollama fallback URL", "https://ollama.com")
OLLAMA_FALLBACK_MODEL = ask("Ollama fallback model", "gemini-3-flash-preview:cloud")

# ── Transcription ──
print("\n── Transcription ──")
USE_YT_TRANSCRIPT_API = ask_bool("Use YouTube Transcript API (skips Whisper)?", False)
USE_DLP_SUBS        = ask_bool("Use yt-dlp auto-captions?", True)
WHISPER_MODEL       = ask("Whisper model", "large-v3")
WHISPER_COMPUTE     = ask("Whisper compute type (float16/float32/int8)", "float16")

# ── Visual Style ──
print("\n── Visual Style ──")
FONT_STYLE    = ask("Font style (DEFAULT/HORMOZI/STORYTELLER/CINEMATIC)", "HORMOZI")
FACE_DETECTOR = ask("Face detector (mediapipe/yolo)", "mediapipe")
YOLO_SIZE     = ask("YOLO size (8n/8s/8m)", "8m")

# ── Content Toggles ──
print("\n── Content Toggles ──")
NO_BGM     = ask_bool("Disable background music?", True)
NO_BROLL   = ask_bool("Disable B-roll?", True)
NO_SUBS    = ask_bool("Disable subtitles?", False)
NO_KARAOKE = ask_bool("Disable karaoke effect?", False)
NO_HOOK    = ask_bool("Disable hook glitch?", False)

# ── Podcast / Advanced ──
print("\n── Podcast / Advanced ──")
SPLIT_SCREEN  = ask_bool("Enable split-screen mode?", False)
SPLIT_TRIGGER = ask("Split trigger (diarization/face)", "diarization")
CAMERA_SWITCH = ask_bool("Enable camera-switch mode?", False)
HOOK_V2       = ask_bool("Enable Hook V2 (multi-hook intro)?", False)
NO_SEGMENT_TRIM = ask_bool("Disable segment trimming?", False)
SILENCE_TRIM  = ask_bool("Aggressively trim silence?", False)

# ── Video Quality ──
print("\n── Video Quality ──")
VIDEO_PRESET     = ask("Encoder preset (ultrafast/fast/slow/veryslow)", "ultrafast")
VIDEO_SCALE_ALGO = ask("Scale algorithm (bicubic/lanczos/bilinear/area)", "bicubic")
VIDEO_CQ         = ask_int("NVENC CQ (lower=sharper)", 23, 15, 50)
VIDEO_CRF        = ask_int("x264 CRF (lower=sharper)", 20, 15, 50)
VIDEO_SHARPEN    = ask_bool("Apply sharpening filter?", False)

# ── Colab Optimizations ──
print("\n── Colab Optimizations ──")
COLAB_CLEANUP = ask_bool("Enable aggressive VRAM cleanup between stages?", True)

# ── Per-Clip Overrides (optional) ──
print("\n── Per-Clip Overrides (optional) ──")
CLIP_OVERRIDES = None
if ask_bool("Configure per-clip overrides?", False):
    # Simple example override
    CLIP_OVERRIDES = {
        "1": {"ratio": "9:16", "font_style": "HORMOZI"},
        "2": {"ratio": "1:1", "font_style": "CINEMATIC"},
    }
    print("⚙️ Using default per-clip overrides. Edit cell to customize.")

CLIP_CONFIG_PATH = None
if CLIP_OVERRIDES:
    CLIP_CONFIG_PATH = str(Path(CLONE_DIR) / "clip_override.json")
    Path(CLIP_CONFIG_PATH).write_text(json.dumps(CLIP_OVERRIDES, indent=2), encoding="utf-8")
    print(f"🎛️ Per-clip overrides written → {CLIP_CONFIG_PATH}")

print("\n✅ Configuration ready. Proceed to Cell 3.")

## ③ RUNTIME — Execute Pipeline

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — RUNTIME
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, time
from pathlib import Path

cmd = [
    sys.executable, "main.py",
    "--url", URL_VIDEO,
    "--source", SOURCE_PLATFORM,
    "--clips", str(JUMLAH_CLIP),
    "--ratio", RASIO,
    "--render-height", str(RENDER_HEIGHT),
    "--source-height", str(SOURCE_HEIGHT),
    "--font-style", FONT_STYLE,
    "--face-detector", FACE_DETECTOR,
    "--yolo-size", YOLO_SIZE,
    "--ai-provider", AI_PROVIDER,
    "--gemini-model", GEMINI_MODEL,
    "--ollama-url", OLLAMA_URL,
    "--ollama-model", OLLAMA_MODEL,
    "--ollama-fallback-url", OLLAMA_FALLBACK_URL,
    "--ollama-fallback-model", OLLAMA_FALLBACK_MODEL,
    "--whisper-model", WHISPER_MODEL,
    "--whisper-compute-type", WHISPER_COMPUTE,
    "--video-preset", VIDEO_PRESET,
    "--video-scale-algo", VIDEO_SCALE_ALGO,
    "--video-cq", str(VIDEO_CQ),
    "--video-crf", str(VIDEO_CRF),
    "--words-per-sub", str(5),
    "--hook-duration", str(3),
]

if COLAB_CLEANUP:
    cmd.append("--colab-cleanup")
if USE_YT_TRANSCRIPT_API:
    cmd.append("--use-yt-transcript-api")
if USE_DLP_SUBS:
    cmd.append("--use-dlp-subs")
if NO_BGM:
    cmd.append("--no-bgm")
if NO_BROLL:
    cmd.append("--no-broll")
if NO_SUBS:
    cmd.append("--no-subs")
if NO_KARAOKE:
    cmd.append("--no-karaoke")
if NO_HOOK:
    cmd.append("--no-hook")
if SPLIT_SCREEN:
    cmd.append("--split-screen")
    cmd += ["--split-trigger", SPLIT_TRIGGER]
if CAMERA_SWITCH:
    cmd.append("--camera-switch")
if HOOK_V2:
    cmd.append("--hook-v2")
if NO_SEGMENT_TRIM:
    cmd.append("--no-segment-trim")
if SILENCE_TRIM:
    cmd.append("--silence-trim")
if VIDEO_SHARPEN:
    cmd.append("--video-sharpen")
if CLIP_CONFIG_PATH:
    cmd += ["--clip-config", CLIP_CONFIG_PATH]

print("🚀 Starting pipeline…")
print("Command:", " ".join(cmd))
print("─" * 60)

start = time.time()
result = subprocess.run(cmd, cwd=CLONE_DIR)
if result.returncode != 0:
    print(f"❌ Pipeline exited with code {result.returncode}")
elapsed = time.time() - start

print("─" * 60)
print(f"⏱️ Total time: {elapsed/60:.1f} minutes")

out_dir = Path(CLONE_DIR) / "outputs"
if out_dir.exists():
    files = sorted(out_dir.glob("*_ready.mp4")) + sorted(out_dir.glob("*.jpg"))
    print(f"\n📦 Output files ({len(files)}):")
    for f in files:
        size_mb = f.stat().st_size / (1024*1024)
        print(f"   • {f.name} ({size_mb:.1f} MB)")
else:
    print("⚠️ outputs/ directory not found.")

# ── Zip outputs ──
if out_dir.exists() and list(out_dir.iterdir()):
    zip_path = Path(CLONE_DIR) / "outputs.zip"
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(out_dir))
    print(f"\n🗜️ outputs.zip → {zip_path.stat().st_size/1024/1024:.1f} MB")
    from google.colab import files
    files.download(str(zip_path))

# ── Persist models to Drive cache ──
copied = 0
for m in ["blaze_face_full_range.tflite", "face_yolov8m.pt", "face_yolov8n.pt", "face_yolov8s.pt"]:
    src = Path(CLONE_DIR) / m
    dst = DRIVE_CACHE / m
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        copied += 1
        print(f"💾 Cached {m} → Drive")
if copied:
    print(f"✅ {copied} model(s) cached to Drive for next session.")

## 🚨 Emergency Cleanup (run if disk almost full)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# EMERGENCY CLEANUP — free temp files instantly
# ═══════════════════════════════════════════════════════════════
import shutil
from pathlib import Path

patterns = ["*.ts", "*_silent_*.mp4", "*.ass", "temp_broll_*", "video_asli.mp4", "*.wav", "bgm_*.mp3"]
removed = sum(1 for pat in patterns for fp in Path(CLONE_DIR).glob(pat) if fp.unlink() or True)
print(f"🗑️ Removed {removed} temp files.")
stat = shutil.disk_usage("/content")
print(f"💾 Disk: {stat.free/1024**3:.1f} GB free / {stat.total/1024**3:.1f} GB total")